In [19]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load Odisha Master CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/OD_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)

features = ['NDVI', 'NDMI', 'NBR', 'BSI']
df = df[['latitude', 'longitude', 'year'] + features]
df = df.dropna()
df = df.sort_values(by=['latitude', 'longitude', 'year'])

# ===============================
# 3. Create Sequences (RAW DATA)
# ===============================
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

required_years = [2021, 2022, 2023, 2024]

for (lat, lon), group in df.groupby(['latitude', 'longitude']):
    years_present = group['year'].tolist()

    # Only include points that have all required years
    if set(required_years).issubset(years_present):
        # Ensure ordering by year
        group_sorted = group.sort_values('year')
        X_raw.append(group_sorted[features].values)
        locations.append([lat, lon])
    else:
        print(f"Skipping location ({lat}, {lon}), missing years: {sorted(set(required_years) - set(years_present))}")

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)
if X_raw.shape[0] == 0:
    raise ValueError("No locations with complete 2021–2024 data. Check your CSV.")

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/OD_Deforestation_Predictions.csv',
    index=False
)

print("Saved: OD_Deforestation_Predictions.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw input shape: (35, 5, 4)

Label distribution:
No deforestation: 33
Deforestation: 2
Train shape: (28, 5, 4)
Test shape : (7, 5, 4)
Scaled train shape: (28, 5, 4)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.1786 - loss: 1.2414 - precision: 0.0800 - recall: 1.0000 - val_accuracy: 0.0000e+00 - val_loss: 0.7122 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.1429 - loss: 1.2361 - precision: 0.0769 - recall: 1.0000 - val_accuracy: 0.0000e+00 - val_loss: 0.7036 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.3214 - loss: 1.2477 - precision: 0.0526 - recall: 0.5000 - val_accuracy: 0.0000e+00 - val_loss: 0.6973 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.3214 - loss: 1.2371 - precision: 0.0526 - recall: 0.5000 - val_accuracy: 0.8571 - val_loss: 0.6899 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.5357 - loss: 1.2154 - precision: 0.1333 - recall: 1.0000 - val_acc

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [20]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/OD_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/OD_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/OD_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.14440250342857144), 'std': 0.05511449849255359, 'q1': np.float64(-0.172174955), 'q3': np.float64(-0.11338471500000002), 'lower_2std': np.float64(-0.25463150041367866), 'upper_2std': np.float64(-0.03417350644346426)}
NBR : {'mean': np.float64(-0.15907586971428572), 'std': 0.06862875624090714, 'q1': np.float64(-0.19800036), 'q3': np.float64(-0.11494624), 'lower_2std': np.float64(-0.2963333821961), 'upper_2std': np.float64(-0.02181835723247144)}
BSI : {'mean': np.float64(0.07909314190411139), 'std': 0.04218176896119433, 'q1': np.float64(0.04857097842286494), 'q3': np.float64(0.09853334447836015), 'lower_2std': np.float64(-0.005270396018277265), 'upper_2std': np.float64(0.16345667982650003)}
NDMI: {'mean': np.float64(-0.06626796514285715), 'std': 0.04837163485313927, 'q1': np.float64(-0.08567637999999998), 'q3': np.float64(-0.03609778999999999), 'lower_2std': np.float64(-0.1630112348491357), 'upper_2std': np.float64(0.030475304563421388)}

Deforestation Cause S

In [22]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Andhra Pradesh map
# AP roughly: latitude 12.6–19.9, longitude 76.8–84.8
m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/OD_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


cause
Agriculture    4
Urban/Other    2
Logging        1
Name: count, dtype: int64


In [23]:
m.save('/content/drive/MyDrive/OD_adaptive.html')


In [24]:
m

In [25]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/OD_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/OD_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD CHHATTISGARH DISTRICTS
# ==============================

cg = gpd.read_file('/content/drive/MyDrive/OD.geojson')

# CRS SAFETY
if cg.crs is None:
    cg.set_crs("EPSG:4326", inplace=True)

cg = cg.to_crs("EPSG:4326")

# Optional label
cg["state"] = "Chhattisgarh"

print(cg.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 7
    latitude  longitude  deforestation
0  21.076722  83.664145              1
1  19.775064  85.624269              1
2  21.768425  86.449821              1
3  20.378731  86.722908              1
4  19.917896  85.168823              1
Total deforestation samples: 7
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  21.076722  83.664145              1    -0.025494   -0.035608    0.030512   
1  19.775064  85.624269              1    -0.212896   -0.189325    0.089891   
2  21.768425  86.449821              1    -0.135576   -0.114046    0.050274   
3  20.378731  86.722908              1    -0.174093   -0.277964    0.167615   
4  19.917896  85.168823              1    -0.197876   -0.175887    0.086013   

   NDMI_change        cause  
0    -0.013044  Urban/Other  
1    -0.069423  Agriculture  
2    -0.042264  Urban/Other  
3    -0.152459      Logging  
4    -0.073573  Agriculture  
  REMARKS_2 Country State_Name State_Code  Dist_Name Dist_Code 

In [26]:
# ==============================
# SPATIAL JOIN
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    cg,
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — Chhattisgarh
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/OD_District_Wise_Deforestation.csv',
    index=False
)

print("Saved CG district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/OD_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved CG cause summary")

    latitude  longitude  deforestation        cause  \
0  21.076722  83.664145              1  Urban/Other   
1  19.775064  85.624269              1  Agriculture   
2  21.768425  86.449821              1  Urban/Other   
3  20.378731  86.722908              1      Logging   
4  19.917896  85.168823              1  Agriculture   

                    geometry  index_right REMARKS_2 Country State_Name  \
0  POINT (83.66414 21.07672)           28      None   India     Orissa   
1  POINT (85.62427 19.77506)           25      None   India     Orissa   
2  POINT (86.44982 21.76843)           21      None   India     Orissa   
3  POINT (86.72291 20.37873)           16      None   India     Orissa   
4   POINT (85.16882 19.9179)           23      None   India     Orissa   

  State_Code   Dist_Name Dist_Code         state  
0         21  Subarnapur       392  Chhattisgarh  
1         21        Puri       387  Chhattisgarh  
2         21  Mayurbhanj       376  Chhattisgarh  
3         21  Kendra

In [27]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Odisha District Boundaries
# =====================================================
od = gpd.read_file('/content/drive/MyDrive/OD.geojson')

# CRS safety
if od.crs is None:
    od.set_crs(epsg=4326, inplace=True)

od = od.to_crs(epsg=4326)
od["state"] = "Odisha"
gdf_districts = od.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/OD_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0  # placeholder if no data

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join (assign districts to points)
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats into District Map
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="Dist_Name", how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

# Deforestation
gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)
gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)
gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

# Afforestation
gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)
gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)
gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (center Odisha)
# =====================================================
m = folium.Map(location=[20.5, 85.9], zoom_start=6)

# =====================================================
# STEP 10: Dynamic Choropleth for Deforestation
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Odisha)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Odisha Afforestation Alert Popup
# =====================================================
state_total = gdf_districts["total_samples"].sum()
state_def = gdf_districts["deforestation_samples"].sum()
state_aff = gdf_districts["afforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Odisha Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Focus on plantation, controlled logging,<br>
and sustainable land management.
"""

folium.Marker(
    location=[20.5, 85.9],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/od_forest.html")

print("✅ Odisha map saved successfully with afforestation recommendation!")

/tmp/ipython-input-270/2246932662.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Odisha map saved successfully with afforestation recommendation!
